In [203]:
import optbinning
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, precision_recall_curve, auc
from sklearn.feature_selection import f_classif
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import chi2_contingency
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
from tqdm import tqdm



from datetime import datetime

In [204]:
df = pd.read_excel('fin_data_of_proj.xlsx')

In [205]:
df

,gara_id,application_name,author_id,builder_id,contractor_id,designer_id,engine_id,manager_id,status,decision_date,region_name,city_id,type,turn,network,period_month,start_date,end_date,class,flat_area,build_qty,build_floor,build_area,com_area,park_area,flat_qty,com_qty,park_qty,flat_onerm,flat_tworm,flat_threerm,flat_fourrm,cost_plan,flat_msprice,build_mscost,flat_mscost,com_msprice,park_price,self_cost,unfin_cost,share_cost,warr_cost,comm_cost,cost_official,flat_inner,flat_height,dpg_date,dpg_num,dpg_add_num,dpg_num_text,prolongation,update_date,state_id,create_date,annul_date,state_name,дата фин данных,bin,date,Итого краткосрочные активы,Итого краткосрочных обязательств,Денежные средства,Краткосрочная дебиторская задолженность,Запасы,Себестоимость,Доход от реализации,Итого активы,Итого капитал,Итого пассивы,Краткосрочная кредиторская задолженность,Операционная прибыль,Долгосрочные финансовые обязательства,Краткосрочные финансовые обязательства,Итоговая прибыль (убыток),Расходы на финансирование,Коэффициент текущей ликвидности,Коэффициент быстрой ликвидности,Коэффициент абсолютной ликвидности,Коэффициент оборачиваемости запасов,Коэффициент оборачиваемости активов,Коэффициент оборачиваемости кредиторской задолженности,ROA,ROE,Коэффициент чистой прибыли,Коэффициент финансового левериджа,Коэффициент финансовой независимости,Коэффициент долговой нагрузки,Коэффициент долг на капитал,Коэффициент покрытия процентов,Покрытие долга,Target,life_cycle,today,real_life_cycle,percent_of_life_cycle,Target 2
0,729352,Заявка на гарантирование - 335,5.400012e+08,960340000376,9.603400e+11,0.000000e+00,2.210400e+11,1.904400e+11,NaN,NaT,г. Астана,18178,Стандартный,"5, 7",будут выполнены за счет застройщика,13.0,2022-11-03,2023-12-03,4,9633.70,NaN,9.0,9633.70,7631.20,313.30,96,2.0,NaN,16.0,34.0,32.0,NaN,2228233,360000.0,231.0,231295.0,400000.0,NaN,3.342349e+08,3.316000e+07,1.893998e+09,2.195073e+07,NaN,2228232541,предчистовая,2.7,2023-06-21,58,7.0,ДС7 21.06.23 к ДПГ-21-01-001/058,NaN,2024-06-04 14:03:30.466878,15320.0,2024-03-08 16:16:36.330524,NaN,Актуальная гарантия,2022-12-31,9.603400e+11,2022-12-31,1.694698e+08,5.837245e+07,1358342.81,6.282570e+06,1.189770e+07,3.297099e+07,6.312265e+07,5.893633e+08,5.063758e+08,5.893633e+08,2.368492e+07,1.062896e+08,2.308623e+07,7384225.12,1.217043e+08,18539551.20,2.903249,0.130899,0.023270,2.771208,0.107103,1.392067,0.206501,0.240344,1.928060,1.163885,0.859191,1.0,0.060174,5.733126,3.488284,0,395,2024-11-01,729,1.845570,0
1,790750,Заявка на гарантирование - 334,4.044001e+10,960340000376,9.603400e+11,0.000000e+00,2.210400e+11,1.904400e+11,NaN,NaT,г. Астана,18178,Стандартный,NaN,имеются,21.0,2022-11-03,2024-08-03,4,19350.28,NaN,12.0,19350.28,16324.11,1398.43,141,12.0,NaN,26.0,35.0,40.0,NaN,3413044920,330000.0,176382.0,176382.0,330000.0,NaN,5.119567e+08,4.913777e+08,2.901088e+09,2.921667e+07,NaN,3413044920,улучшенная черновая,2.7,2023-08-11,54,9.0,дс9 11.08.23 к ДПГ-21-01-001/054,NaN,2024-06-04 14:03:30.466878,15320.0,2024-03-08 16:16:36.330524,NaN,Актуальная гарантия,2022-12-31,9.603400e+11,2022-12-31,1.694698e+08,5.837245e+07,1358342.81,6.282570e+06,1.189770e+07,3.297099e+07,6.312265e+07,5.893633e+08,5.063758e+08,5.893633e+08,2.368492e+07,1.062896e+08,2.308623e+07,7384225.12,1.217043e+08,18539551.20,2.903249,0.130899,0.023270,2.771208,0.107103,1.392067,0.206501,0.240344,1.928060,1.163885,0.859191,1.0,0.060174,5.733126,3.488284,0,639,2024-11-01,729,1.140845,0
2,845821,Заявка на гарантирование - 407,8.034001e+10,60340004172,6.034000e+10,0.000000e+00,1.304400e+11,2.109400e+11,NaN,NaT,г. Шымкент,18277,Стандартный,NaN,иное,9.0,2023-06-12,2024-03-12,4,11723.00,NaN,9.0,11723.00,6650.80,1070.64,80,2.0,1635.0,8.0,40.0,24.0,NaN,2705190883,430000.0,230759.0,430000.0,450000.0,2000000.0,2.972323e+08,2.972323e+08,2.637281e+09,2.637281e+07,NaN,2934513361,чистовая,2.7,2023-09-14,151,1.0,ДПГ-23-13-050/151,NaN,2024-06-04 14:03:30.466878,15320.0,2024-03-08 16:16:36.330524,NaN,Актуальная гарантия,2022-12-31,6.034000e+10,2022-12-

In [206]:
# Unfin_cost and Season of every year

def get_season(date):
    month = date.month if pd.notnull(date) else None
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'
    else:
        return None

df['season'] = df['dpg_date'].apply(get_season)
df['unfin_coff'] = df['unfin_cost']/df['cost_plan']

In [207]:
# Inflaction of every year 8%

annual_inflation_rate = 0.08
inflation_reference_date = datetime(2024, 11, 1)


df['dpg_date'] = pd.to_datetime(df['dpg_date'], errors='coerce')

financial_columns = [
    'cost_plan', 'flat_msprice', 'build_mscost', 'flat_mscost', 'com_msprice', 'park_price',
    'self_cost', 'unfin_cost', 'share_cost', 'warr_cost', 'comm_cost', 'cost_official',
    'Итого краткосрочные активы', 'Итого краткосрочных обязательств', 'Денежные средства',
    'Краткосрочная дебиторская задолженность', 'Запасы', 'Себестоимость', 'Доход от реализации',
    'Итого активы', 'Итого капитал', 'Итого пассивы', 'Краткосрочная кредиторская задолженность',
    'Операционная прибыль', 'Долгосрочные финансовые обязательства', 'Краткосрочные финансовые обязательства',
    'Итоговая прибыль (убыток)', 'Расходы на финансирование'
]

df['years_since_dpg'] = (inflation_reference_date - df['dpg_date']).dt.days / 365.25
df['inflation_factor'] = (1 + annual_inflation_rate) ** df['years_since_dpg']

for col in financial_columns:
    if col in df.columns:
        df[col] = df[col] * df['inflation_factor']

df.drop(columns=['years_since_dpg', 'inflation_factor'], inplace=True)


In [208]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

df.replace([np.inf, -np.inf], np.nan, inplace=True)

df = df[:-11]
df = df.drop(columns=['application_name', 'author_id', 'engine_id', 'manager_id', 'status', 'decision_date',
                          'region_name', 'build_mscost', 'dpg_num_text', 'prolongation', 'update_date', 'state_id',
                          'create_date', 'annul_date', 'state_name', 'дата фин данных', 'life_cycle', 'today',
                          'gara_id', 'contractor_id', 'designer_id', 'build_qty', 'real_life_cycle', 'bin', 
                          'date', 'start_date', 'end_date', 'turn', 'network', 'Коэффициент долговой нагрузки', 
                          'dpg_add_num', 'dpg_num', 'dpg_date'])

df['coff_doxoda_ot_real'] = df['Доход от реализации'] / df['cost_plan']

df = df[df['type'] != 'Паркинг']

df = df.drop(columns=['type'])


df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.drop('Target 2', axis = 1)
df = df.dropna(axis=1, how='all')

non_numeric_cols = df.columns[(df.nunique() < 40) & (df.columns != 'Target')]

label_encoder = LabelEncoder()
for col in non_numeric_cols:
    df[col] = label_encoder.fit_transform(df[col].astype(str))



In [209]:
df.columns

Index(['builder_id', 'city_id', 'period_month', 'class', 'flat_area',
       'build_floor', 'build_area', 'com_area', 'park_area', 'flat_qty',
       'com_qty', 'park_qty', 'flat_onerm', 'flat_tworm', 'flat_threerm',
       'flat_fourrm', 'cost_plan', 'flat_msprice', 'flat_mscost',
       'com_msprice', 'park_price', 'self_cost', 'unfin_cost', 'share_cost',
       'warr_cost', 'comm_cost', 'cost_official', 'flat_inner', 'flat_height',
       'Итого краткосрочные активы', 'Итого краткосрочных обязательств',
       'Денежные средства', 'Краткосрочная дебиторская задолженность',
       'Запасы', 'Себестоимость', 'Доход от реализации', 'Итого активы',
       'Итого капитал', 'Итого пассивы',
       'Краткосрочная кредиторская задолженность', 'Операционная прибыль',
       'Долгосрочные финансовые обязательства',
       'Краткосрочные финансовые обязательства', 'Итоговая прибыль (убыток)',
       'Расходы на финансирование', 'Коэффициент текущей ликвидности',
       'Коэффициент быстрой лик

In [210]:
df.to_excel('df.xlsx')

In [211]:
pd.set_option('display.max_columns', None)
if df[col].dtype == 'object':
    df[col] = df[col].str.replace(',', '')
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

In [212]:
df.replace(np.NaN, 0, inplace=True)

In [213]:
df[non_numeric_cols] = df[non_numeric_cols].astype(str)

In [214]:
iv_score_dict = {}
for col in tqdm(X.columns):
    if col in cat_cols:
        optb = optbinning.OptimalBinning(dtype='categorical')
        optb.fit(df[col], df['Target'])
    else:
        optb = optbinning.OptimalBinning(dtype='numerical')
        optb.fit(df[col], df['Target'])
    binning_table = optb.binning_table
    binning_table.build()
    iv_score_dict[col] = binning_table.iv
iv_score_df = pd.Series(iv_score_dict)
iv_score_df.sort_values(ascending=False, inplace=True)

100%|██████████████████████████████████████████████████████████████████████████████████| 63/63 [00:03<00:00, 16.16it/s]


In [215]:
pd.set_option('display.max_columns', None)

In [216]:
iv_score_df

Коэффициент оборачиваемости активов       1.283359
period_month                              1.175856
city_id                                   0.954683
share_cost                                0.937570
Итого краткосрочные активы                0.612790
                                            ...   
self_cost                                 0.131563
Краткосрочные финансовые обязательства    0.126889
Коэффициент долг на капитал               0.082434
park_price                                0.074951
class                                     0.020817
Length: 63, dtype: float64

In [217]:
iv_score_df2 = pd.DataFrame(iv_score_df, columns=['Numbers']).reset_index()
iv_score_df3 = iv_score_df2[iv_score_df2['Numbers'].between(0.5, 2)]
selected_features = iv_score_df3['index']
selected_features

0          Коэффициент оборачиваемости активов
1                                 period_month
2                                      city_id
3                                   share_cost
4                   Итого краткосрочные активы
5                        percent_of_life_cycle
6                                 Итого активы
7                                Итого пассивы
8                    Расходы на финансирование
9                                      com_qty
10    Краткосрочная кредиторская задолженность
11                                         ROE
12                                 build_floor
13                        Операционная прибыль
14                                         ROA
Name: index, dtype: object

In [218]:
df2 = pd.concat([df[selected_features], df['Target']], axis = 1)
X = df2.drop(['Target'], axis = 1)
y = df2['Target']
X_train, y_train, X_test, y_test = train_test_split(X, y, test_size = 0.35, random_state = 42)

In [219]:
from sklearn import linear_model

binning_process = optbinning.BinningProcess(
  variable_names=X.columns.to_list())
# estimator
estimator = LogisticRegression()
scorecard = optbinning.Scorecard(
  binning_process=binning_process,
  estimator=estimator,
  scaling_method="min_max",
  scaling_method_params={"min": 300, "max": 850},

)

In [221]:
scorecard.fit(X, y)

Scorecard(binning_process=BinningProcess(variable_names=['Коэффициент '
                                                         'оборачиваемости '
                                                         'активов',
                                                         'period_month',
                                                         'city_id',
                                                         'share_cost',
                                                         'Итого краткосрочные '
                                                         'активы',
                                                         'percent_of_life_cycle',
                                                         'Итого активы',
                                                         'Итого пассивы',
                                                         'Расходы на '
                                                         'финансирование',
                                                         'com_qty',
                                                         'Краткосрочная '
                                                         'кредиторская '
                                                         'задолженность',
                                                         'ROE', 'build_floor',
                                                         'Операционная прибыль',
                                                         'ROA']),
          estimator=LogisticRegression(), scaling_method='min_max',
          scaling_method_params={'max': 850, 'min': 300})

In [222]:
scorecard.intercept_

0

In [223]:
scorecard_df = scorecard.table(style="detailed")

In [224]:
len(scorecard_df['Variable'].unique())

15

In [225]:
scorecard_df.to_excel('scorecard.xlsx')

In [226]:
prediction = scorecard.predict_proba(X)[:, 1]
print(f'Logit ROC_AUC: {roc_auc_score(y, prediction )}')

Logit ROC_AUC: 0.980186702228996


In [227]:
scorecard_df2 = scorecard_df.drop_duplicates(subset=['Variable'])
scorecard_df2= scorecard_df2[['Variable', 'Coefficient']]
scorecard_df2

,Variable,Coefficient
0,Коэффициент оборачиваемости активов,-0.541506
0,period_month,-0.527451
0,city_id,-1.180631
0,share_cost,-0.893418
0,Итого краткосрочные активы,-0.387786
0,percent_of_life_cycle,-0.541681
0,Итого активы,-0.421593
0,Итого пассивы,-0.421593
0,Расходы на финансирование,-0.193153
0,com_qty,-0.454107
